In [ ]:
import os
import pandas as pd
import pickle
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# NEW data - on each fund separetely

In [ ]:
path = r'../../Data/Each fund assets/5523_291021_if_form_766.xls'
data = pd.read_excel(path, engine = 'xlrd')

res = xls_form_process(data)

In [ ]:
def xls_form_process(data):
    # Delete the first column
    data = data.iloc[:, 1:]
    # Use the 10th row (index 9) as the header
    new_header = data.iloc[10]
    data = data[:]
    data.columns = new_header
    data = data.drop(data.index[10])
    # Extract the value from cell at row index 1 and column index 2 (i.e., 1st row, 3rd column)
    full_text = data.iloc[2, 0] 
    prefix = "Состав активов и обязательств фонда "
    # Extract everything after the prefix and remove extra whitespace
    name_value = full_text.split(prefix)[-1].strip()

    # Create a new column 'name' with this value for all rows
    if len(name_value) > 1:
        name_value = name_value[1:-1]
    # Create a new column 'name' with this value for all rows
    data['Название фонда'] = name_value
    data['Название УК'] = data.iloc[5, 1]
    data["date"] = pd.to_datetime(data.iloc[3, 1], format="%d.%m.%Y")
    data["date"] = data["date"].dt.strftime("%Y-%m-%d")
    data['Номер регистрации'] = data.iloc[4, 1]
    data['Тип фонда'] = data.iloc[6, 1].split(', ')[0]
    data['Категория фонда'] =data.iloc[6, 1].split(', ')[1]
    data['СЧА на дату отчетности, руб.'] = data.iloc[7, 1]
    data['Итого сумма активов, тыс. руб.'] = data.iloc[-6, 2]
    data['Итого сумма обязательств, тыс. руб.'] = data.iloc[-1, 1]
    data = data.loc[11:].reset_index(drop=True)
    data = data.iloc[:-6]
    subclass_ind = data.index[data.iloc[:, 0].astype(str).str.startswith("- ")].tolist()

    groups = np.split(np.array(subclass_ind), np.where(np.diff(subclass_ind) != 1)[0] + 1)
    mask = data["Регистрационный номер ценной бумаги"].notna() & (data["Регистрационный номер ценной бумаги"] != "")
    class_indices = []
    for grp in groups:
        candidate = grp[0] - 1
        class_indices.append(candidate)
        seq = data.index[mask & (data.index >= candidate + 1)]
        if len(seq) > 0:
            seq_groups = np.split(np.array(seq), np.where(np.diff(seq) != 1)[0] + 1)
            first_seq = seq_groups[0]
            candidate_before = first_seq[0] - 1
            candidate_after = first_seq[-1] + 1
            class_indices.extend([candidate_before, candidate_after])
    final_class_indices = sorted(set(class_indices))
    boundaries = np.array(final_class_indices)  
    labels = data.iloc[boundaries, 0].to_numpy()  
    pos = np.searchsorted(boundaries, data.index, side='right') - 1  
    data.loc[:, 'Класс активов'] = labels[pos]
    subclass_arr = np.array(subclass_ind)
    labels = data.iloc[subclass_arr, 0].str.replace("^- ", "", regex=True).to_numpy()
    pos = np.searchsorted(subclass_arr, data.index, side="right") - 1
    data["Подкласс активов"] = [labels[i] if i >= 0 else "" for i in pos]
    data = data.drop(index=final_class_indices)
    to_drop = [i-1 for i in data.index 
            if pd.notna(data.loc[i, 'Регистрационный номер ценной бумаги']) 
            and (i-1 in data.index and pd.isna(data.loc[i-1, 'Регистрационный номер ценной бумаги']))]
    data = data.drop(index=to_drop)
    data = data[~data["Название вида актива или ценной бумаги"].str.contains("итого", case=False, na=False)]
    return(data)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

folder_path = r'../../Data/Each fund assets'
all_dfs = []
unprocessed_files = []

for fname in tqdm(os.listdir(folder_path), desc="Processing XLS files"):
    if fname.lower().endswith('.xls'):
        fpath = os.path.join(folder_path, fname)
        try:
            data = pd.read_excel(fpath, engine='xlrd')
            res = xls_form_process(data)
            all_dfs.append(res)
        except Exception as e:
            print(f"Error processing {fname}: {e}")
            unprocessed_files.append(fname)

if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_excel(r'../../Data/processed_all.xlsx', index=False)

# ONE LONH DF

In [ ]:
# with open(r'../../Data/Processed_data/funds_database_1layer_wide_distinct.pkl', "rb") as f:
#     loaded_db = pickle.load(f)

In [ ]:
# with open(r'../../Data/Processed_data/funds_database_1layer_wide_distinct_values_shares.pkl', "wb") as f:
#     pickle.dump(dict_of_df, f)

## LONG DF

In [ ]:
df = pd.read_csv(r'../../Data/Processed_data/aggregated.csv', encoding='UTF-16')

## DICT of df создание словаря с широкими датафреймами

In [ ]:
df['key'] = df["Название фонда"]+'__'+df["Номер регистрации"]

In [ ]:
q = df[['Название фонда','ISIN Фонда']].drop_duplicates().dropna()

In [ ]:
q.to_csv('name-ISIN.csv', index=False, encoding='UTF-16')

In [ ]:
db_structure = {}
for fund in df["key"].unique():
    fund_df = df[df["key"] == fund].drop(columns=["key"]).set_index("date")
    db_structure[fund] = fund_df
del(df)

In [ ]:
dict_of_df = {}
j = 0
for j in range(len(list(db_structure.keys()))):
    try:
        fund_data = db_structure[list(db_structure.keys())[j]]
        slices = {date: fund_data[fund_data.index == date][['Конечный актив','Сумма, руб.']] for date in fund_data.index.unique()}

        merged = slices[list(slices.keys())[0]]
        merged = merged.rename(columns={'Сумма, руб.': str(list(merged.index)[0])})
        for i in list(slices.keys())[1:]:
            add = slices[i].copy()
            add = add.rename(columns={'Сумма, руб.': str(i)})
            merged = pd.merge(merged, add, on='Конечный актив', how='outer').fillna(0)
            
        merged = pd.merge(fund_data.iloc[:,0:-2].drop_duplicates(), merged, on='Конечный актив', how='right')
        # merged.insert(0,'Название фонда',list(db_structure.keys())[j])

        dict_of_df[list(db_structure.keys())[j]] = merged
    except: None

In [ ]:
dict_of_df = {}
j = 0
for j in range(len(list(db_structure.keys()))):
    try:
        fund_data = db_structure[list(db_structure.keys())[j]]
        slices = {date: fund_data[fund_data.index == date][['Конечный актив','Доля, %']] for date in fund_data.index.unique()}

        merged = slices[list(slices.keys())[0]]
        merged = merged.rename(columns={'Доля, %': str(list(merged.index)[0])})
        for i in list(slices.keys())[1:]:
            add = slices[i].copy()
            add = add.rename(columns={'Доля, %': str(i)})
            merged = pd.merge(merged, add, on='Конечный актив', how='outer').fillna(0)
            
        merged = pd.merge(fund_data.iloc[:,0:-2].drop_duplicates(), merged, on='Конечный актив', how='right')
        # merged.insert(0,'Название фонда',list(db_structure.keys())[j])

        dict_of_df[list(db_structure.keys())[j]] = merged
    except: None

In [ ]:
dict_of_df['Альфа-Капитал Управляемые акции__4213']

In [ ]:
dict_of_df['Альфа-Капитал Управляемые акции__4213']['2024-11-29'].sum()

In [ ]:
a =pd.Series(dict_of_df.keys())

# other

In [ ]:
def xls_form_process(data):
    # Delete the first column
    data = data.iloc[:, 1:]
    # Use the 10th row (index 9) as the header
    new_header = data.iloc[10]
    data = data[:]
    data.columns = new_header
    data = data.drop(data.index[10])
    # Extract the value from cell at row index 1 and column index 2 (i.e., 1st row, 3rd column)
    full_text = data.iloc[2, 0] 
    prefix = "Состав активов и обязательств фонда "
    # Extract everything after the prefix and remove extra whitespace
    name_value = full_text.split(prefix)[-1].strip()

    # Create a new column 'name' with this value for all rows
    if len(name_value) > 1:
        name_value = name_value[1:-1]
    # Create a new column 'name' with this value for all rows
    data['Название фонда'] = name_value
    data['Название УК'] = data.iloc[5, 1]
    data["date"] = pd.to_datetime(data.iloc[3, 1], format="%d.%m.%Y")
    data["date"] = data["date"].dt.strftime("%Y-%m-%d")
    data['Номер регистрации'] = data.iloc[4, 1]
    data['Тип фонда'] = data.iloc[6, 1].split(', ')[0]
    data['Категория фонда'] =data.iloc[6, 1].split(', ')[1]
    data['СЧА на дату отчетности, руб.'] = data.iloc[7, 1]
    data['Итого сумма активов, тыс. руб.'] = data.iloc[-6, 2]
    data['Итого сумма обязательств, тыс. руб.'] = data.iloc[-1, 1]
    data = data.loc[11:].reset_index(drop=True)
    data = data.iloc[:-6]
    subclass_ind = data.index[data.iloc[:, 0].astype(str).str.startswith("- ")].tolist()

    groups = np.split(np.array(subclass_ind), np.where(np.diff(subclass_ind) != 1)[0] + 1)
    class_indices = [group[0] - 1 for group in groups]
    boundaries = np.array(class_indices)  
    labels = data.iloc[boundaries, 0].to_numpy()  
    pos = np.searchsorted(boundaries, data.index, side='right') - 1  
    data.loc[:, 'Класс активов'] = labels[pos]
    subclass_arr = np.array(subclass_ind)
    labels = data.iloc[subclass_arr, 0].str.replace("^- ", "", regex=True).to_numpy()
    pos = np.searchsorted(subclass_arr, data.index, side="right") - 1
    data["Подкласс активов"] = [labels[i] if i >= 0 else "" for i in pos]
    data = data.drop(index=class_indices)
    to_drop = [i-1 for i in data.index 
            if pd.notna(data.loc[i, 'Регистрационный номер ценной бумаги']) 
            and (i-1 in data.index and pd.isna(data.loc[i-1, 'Регистрационный номер ценной бумаги']))]
    data = data.drop(index=to_drop)

    return(data)

In [ ]:
dict_of_df = {}
j = 0
for j in range(len(df["Название фонда"].unique()))[30:100]:
    fund_data = df[df["Название фонда"] == df["Название фонда"].unique()[j]].set_index("date")
    slices = {date: fund_data[fund_data.index == date][['Конечный актив','Доля, %']] for date in fund_data.index.unique()}

    merged = slices[list(slices.keys())[0]]
    merged = merged.rename(columns={'Доля, %': str(list(merged.index)[0])})
    for i in list(slices.keys())[1:]:
        add = slices[i].copy()
        add = add.rename(columns={'Доля, %': str(i)})
        merged = pd.merge(merged, add, on='Конечный актив', how='outer').fillna(0)
        
    merged = pd.merge(fund_data.iloc[:,0:-2].drop_duplicates(), merged, on='Конечный актив', how='right')
    # merged.insert(0,'Название фонда',df["Название фонда"].unique()[j])

    dict_of_df[df["Название фонда"].unique()[j]] = merged
    del(merged)
# display(merged)

In [ ]:
df = pd.read_csv(r'../../Data/Processed_data/aggregated.csv', encoding='UTF-16')

In [ ]:
long_df = pd.DataFrame()
j = 0
for j in range(len(df["Название фонда"].unique()))[:30]:
    fund_data = df[df["Название фонда"] == df["Название фонда"].unique()[j]].set_index("date")
    slices = {date: fund_data[fund_data.index == date][['Конечный актив','Доля, %']] for date in fund_data.index.unique()}

    merged = slices[list(slices.keys())[0]]
    merged = merged.rename(columns={'Доля, %': str(list(merged.index)[0])})
    for i in list(slices.keys())[1:]:
        add = slices[i].copy()
        add = add.rename(columns={'Доля, %': str(i)})
        merged = pd.merge(merged, add, on='Конечный актив', how='outer').fillna(0)
        
    merged = pd.merge(fund_data.iloc[:,0:-2].drop_duplicates(), merged, on='Конечный актив', how='right')
    # merged.insert(0,'Название фонда',df["Название фонда"].unique()[j])

    long_df = pd.concat([long_df, merged])
    del(merged, slices, add)
# display(merged)

## INDIV CSV

In [ ]:
def save_as_csv(df,j):
    fund_data = df[df["key"] == df["key"].unique()[j]].set_index("date")
    slices = {date: fund_data[fund_data.index == date][['Конечный актив','Доля, %']] for date in fund_data.index.unique()}

    merged = slices[list(slices.keys())[0]].drop_duplicates()
    merged = merged.rename(columns={'Доля, %': str(list(merged.index)[0])}).drop_duplicates()
    for i in list(slices.keys())[1:]:
        add = slices[i].copy()
        add = add.rename(columns={'Доля, %': str(i)})
        merged = pd.merge(merged, add, on='Конечный актив', how='outer').fillna(0).drop_duplicates()
        
    merged = pd.merge(fund_data.iloc[:,0:-2].drop_duplicates(), merged, on='Конечный актив', how='right')
    merged.to_csv(rf'../../Data/Processed_data/Individual_dynamics/{df["key"].unique()[j]}.csv',index=False, encoding='UTF-16')

for j in range(len(df["key"].unique()))[1138:]:
    try:
        save_as_csv(df,j)
    except: None
    print(j)